<a href="https://colab.research.google.com/github/shashankkumarop-creator/HCL/blob/main/E_Learning_Engagement_Pandas_NumPy_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Learning Engagement Analysis Using Pandas and NumPy

**Google Colab notebook**

This project analyzes learner engagement using a CSV dataset. It uses only **Pandas** and **NumPy** as Python libraries.

### Before running
1. In Google Colab, upload your CSV using the left-side **Files** panel (folder icon → Upload).
2. Keep the uploaded CSV in the Colab working directory.
3. Run the cells from top to bottom.

Expected dataset: 1,000 rows and 12 columns.

In [3]:
import pandas as pd
import numpy as np

## 1. Read the CSV dataset
Replace the filename below with the exact name of your uploaded CSV if it differs.

In [4]:
file_name = '18_elearning_engagement_new_1_to_1000-v2.csv'
df = pd.read_csv(file_name)

print('Dataset loaded successfully')
print('Dataset shape:', df.shape)
display(df.head())

Dataset loaded successfully
Dataset shape: (1000, 12)


,Enrollment_ID,Learner_ID,Course_Name,Course_Difficulty,Course_Length_Hours,Hours_Watched,Quiz_Score,Quiz_Attempts,Forum_Posts,Device_Type,Paid_Tier,Completed
0,EN00001,U0258,Web Development,Advanced,13.5,32.3,41.4,0,11,Desktop,Free,No
1,EN00002,U0391,Cloud Basics,Intermediate,16.4,24.1,23.8,4,18,Mobile,Paid,Yes
2,EN00003,U0078,Web Development,Advanced,36.6,26.8,87.1,9,25,Desktop,Free,Yes
3,EN00004,U0334,Data Analytics,Beginner,32.5,4.3,14.8,4,26,Desktop,Free,Yes
4,EN00005,U0542,SQL Essentials,Intermediate,36.1,38.6,49.5,0,30,Tablet,Paid,No


## 2. Check and prepare the dataset
Check that expected columns exist, select them in the intended order, convert numeric fields, and inspect missing/duplicate values.

In [5]:
expected_columns = [
    "Enrollment_ID", "Learner_ID", "Course_Name", "Course_Difficulty",
    "Course_Length_Hours", "Hours_Watched", "Quiz_Score", "Quiz_Attempts",
    "Forum_Posts", "Device_Type", "Paid_Tier", "Completed"
]

missing_columns = set(expected_columns) - set(df.columns)
if missing_columns:
    print("Missing columns:", sorted(missing_columns))
else:
    print("All required columns are present")

df = df[expected_columns].copy()

numeric_columns = [
    "Course_Length_Hours", "Hours_Watched", "Quiz_Score",
    "Quiz_Attempts", "Forum_Posts"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("\nDataset shape:", df.shape)
print("\nMissing values per column:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Enrollment_ID values:", df["Enrollment_ID"].duplicated().sum())
display(df.head())

All required columns are present

Dataset shape: (1000, 12)

Missing values per column:


,0
Enrollment_ID,0
Learner_ID,0
Course_Name,0
Course_Difficulty,0
Course_Length_Hours,0
Hours_Watched,25
Quiz_Score,25
Quiz_Attempts,0
Forum_Posts,0
Device_Type,0


Duplicate rows: 0
Duplicate Enrollment_ID values: 0


,Enrollment_ID,Learner_ID,Course_Name,Course_Difficulty,Course_Length_Hours,Hours_Watched,Quiz_Score,Quiz_Attempts,Forum_Posts,Device_Type,Paid_Tier,Completed
0,EN00001,U0258,Web Development,Advanced,13.5,32.3,41.4,0,11,Desktop,Free,No
1,EN00002,U0391,Cloud Basics,Intermediate,16.4,24.1,23.8,4,18,Mobile,Paid,Yes
2,EN00003,U0078,Web Development,Advanced,36.6,26.8,87.1,9,25,Desktop,Free,Yes
3,EN00004,U0334,Data Analytics,Beginner,32.5,4.3,14.8,4,26,Desktop,Free,Yes
4,EN00005,U0542,SQL Essentials,Intermediate,36.1,38.6,49.5,0,30,Tablet,Paid,No


## 3. Course-wise enrollment and completion
Shows enrollments, completed enrollments, and average quiz score for each course.

In [6]:
df["Completed_Flag"] = np.where(
    df["Completed"].astype(str).str.strip().str.lower() == "yes", 1, 0
)

course_summary = df.groupby("Course_Name").agg(
    Total_Enrollments=("Enrollment_ID", "count"),
    Completed_Enrollments=("Completed_Flag", "sum"),
    Average_Quiz_Score=("Quiz_Score", "mean")
).reset_index()
course_summary["Average_Quiz_Score"] = np.round(course_summary["Average_Quiz_Score"], 2)
course_summary = course_summary.sort_values("Total_Enrollments", ascending=False)
display(course_summary)

,Course_Name,Total_Enrollments,Completed_Enrollments,Average_Quiz_Score
4,SQL Essentials,197,120,50.28
3,Python Fundamentals,173,103,47.83
0,AI Introduction,172,109,51.61
2,Data Analytics,163,100,49.24
1,Cloud Basics,158,97,50.60
5,Web Development,137,87,46.99


## 4. Completion percentage by device
Compares completion rates across device types.

In [7]:
device_summary = df.groupby("Device_Type").agg(
    Total_Enrollments=("Enrollment_ID", "count"),
    Completed_Enrollments=("Completed_Flag", "sum")
).reset_index()

device_summary["Completion_Percent"] = np.round(
    (device_summary["Completed_Enrollments"] / device_summary["Total_Enrollments"]) * 100, 2
)
device_summary = device_summary.sort_values("Completion_Percent", ascending=False)
display(device_summary)

,Device_Type,Total_Enrollments,Completed_Enrollments,Completion_Percent
2,Tablet,332,218,65.66
0,Desktop,341,207,60.70
1,Mobile,327,191,58.41


## 5. Paid vs free learners
Compares average watched hours, average quiz score, and completion percentage by paid tier.

In [8]:
tier_summary = df.groupby("Paid_Tier").agg(
    Total_Enrollments=("Enrollment_ID", "count"),
    Average_Hours_Watched=("Hours_Watched", "mean"),
    Average_Quiz_Score=("Quiz_Score", "mean"),
    Completed_Enrollments=("Completed_Flag", "sum")
).reset_index()

tier_summary["Average_Hours_Watched"] = np.round(tier_summary["Average_Hours_Watched"], 2)
tier_summary["Average_Quiz_Score"] = np.round(tier_summary["Average_Quiz_Score"], 2)
tier_summary["Completion_Percent"] = np.round(
    (tier_summary["Completed_Enrollments"] / tier_summary["Total_Enrollments"]) * 100, 2
)
display(tier_summary)

,Paid_Tier,Total_Enrollments,Average_Hours_Watched,Average_Quiz_Score,Completed_Enrollments,Completion_Percent
0,Free,484,20.99,49.38,289,59.71
1,Paid,516,20.94,49.64,327,63.37


## 6. Top 10 learners by total watched hours
Ranks learners by total hours watched across their enrollments.

In [9]:
learner_activity = df.groupby("Learner_ID").agg(
    Total_Hours_Watched=("Hours_Watched", "sum"),
    Total_Enrollments=("Enrollment_ID", "count")
).reset_index()

learner_activity["Total_Hours_Watched"] = np.round(learner_activity["Total_Hours_Watched"], 2)
learner_activity = learner_activity.sort_values("Total_Hours_Watched", ascending=False).head(10)
display(learner_activity)

,Learner_ID,Total_Hours_Watched,Total_Enrollments
388,U0522,180.6,6
185,U0246,164.2,5
174,U0230,154.1,7
473,U0641,115.5,4
32,U0039,114.6,4
134,U0179,109.9,4
463,U0628,108.0,3
357,U0479,104.8,4
351,U0471,101.1,3
108,U0143,100.9,3


## 7. Save analysis outputs (optional)
Save the summary tables as CSV files in the Colab working directory. No additional libraries are needed.

In [10]:
course_summary.to_csv('course_summary.csv', index=False)
device_summary.to_csv('device_summary.csv', index=False)
tier_summary.to_csv('paid_free_summary.csv', index=False)
learner_activity.to_csv('top_10_learners.csv', index=False)
print('Summary CSV files saved in the Colab working directory.')

Summary CSV files saved in the Colab working directory.
